# Analytical DataFrame API Queries over the Trossen Catalog

The same catalog questions as `analyze_sql.ipynb`, expressed with the DataFusion
**DataFrame API** (`col` / `functions as F`) instead of SQL strings. Start
`pixi run serve` and `pixi run register` first.

In [ ]:
from __future__ import annotations

import os

import pyarrow as pa
import rerun as rr
from datafusion import col, lit
from datafusion import functions as F
from rerun.notebook import Viewer

from trossen_oss.catalog import DEFAULT_CATALOG_URL

url = os.getenv("RERUN_CLOUD_ADDR", DEFAULT_CATALOG_URL)
dataset_name = os.getenv("DATASET_NAME", "trossen_oss")

client = rr.catalog.CatalogClient(url)
dataset = client.get_dataset(dataset_name)

TIMELINE = "message_log_time"

## Segment-level analysis (DataFrame API)

`segment_overview` is itself a DataFrame-API query (`select` + `sort`); reuse it
for the per-episode inventory.

In [ ]:
from trossen_oss.query import segment_overview

segment_overview(dataset)

In [ ]:
Viewer(url=url, width="auto", height=300)

## Content-level analysis (DataFrame API)

Per-episode sample count and travel (`max − min`) for one joint, built with the
DataFrame API. `filter_contents` reduces the Scalars column to a one-element
`list<double>`; `array_element(col, 1)` pulls the value (1-indexed, like SQL `[1]`).

In [ ]:
joint = "/robot_right/joints/joint_1"
value_column = f"{joint}:Scalars:scalars"

df = dataset.filter_contents([joint]).reader(index=TIMELINE)
value = F.array_element(col(value_column), lit(1)).alias("value")

# DataFusion's DataFrame aggregate() takes pure aggregate functions, so we emit
# max/min as columns and subtract them in a follow-up select.
stats = (
    df.select("rerun_segment_id", value)
    .aggregate(
        [col("rerun_segment_id")],
        [
            F.count(col("value")).alias("num_samples"),
            F.max(col("value")).alias("v_max"),
            F.min(col("value")).alias("v_min"),
        ],
    )
    .select("rerun_segment_id", "num_samples", (col("v_max") - col("v_min")).alias("travel"))
    .sort(col("travel").sort(ascending=False))
)
pa.table(stats).slice(0, 20)

## Filter to the interesting segments

A small candidate set for Viewer inspection — episodes where this joint moves
more than 1 rad.

In [ ]:
interesting = stats.filter(col("travel") > 1.0).select("rerun_segment_id", "travel")
pa.table(interesting).slice(0, 20)